### This notebook format data for training deepSCENIC from a SCENIC+ object

In [1]:
import pickle
import scanpy as sc
import numpy as np
import pandas as pd
import os

Read the SCENIC+ object

In [2]:
import dill
scplus_folder = "/staging/leuven/stg_00002/lcb/sdewin/PhD/Multiomics_pipeline/analysis/pbmc_granulocyte_sorted_10k/scenicplus_clustered_consensus"
scenicpls_obj = dill.load(
        open(os.path.join(scplus_folder, 'scplus_obj.pkl'), 'rb'))

In [3]:
# PATH WHERE TO SAVE THE DATA
DATA_PATH = '/staging/leuven/stg_00002/lcb/gpartel/projects/deepSCENIC/data/PBMC/data/deepSCENIC/'

# Save cell metadata
scenicpls_obj.metadata_cell.to_csv(DATA_PATH+'cells_metadata.csv')

Extract RNA and ATAC data, preprocessing and filtering

In [4]:
ad_rna = sc.AnnData(scenicpls_obj.to_df('EXP'))
ad_rna

AnnData object with n_obs × n_vars = 9409 × 26346

In [5]:
sc.pp.filter_genes(ad_rna, min_cells=10)
ad_rna

AnnData object with n_obs × n_vars = 9409 × 20827
    var: 'n_cells'

In [6]:
sc.pp.normalize_total(ad_rna)
sc.pp.log1p(ad_rna)
ad_rna.layers['log_norm'] = ad_rna.X

In [7]:
# Filter regions based on minimum number of fragments
min_nr_acc = 20
scenicpls_obj.subset(regions = scenicpls_obj.metadata_regions[scenicpls_obj.metadata_regions.cisTopic_nr_acc>min_nr_acc].index.to_list())

In [8]:
ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T)
ad_atac

/tmp/ipykernel_7818/3854231756.py:1: FutureWarning: X.dtype being converted to np.float32 from int32. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T)


AnnData object with n_obs × n_vars = 9409 × 323145

Get regions per gene (up to 150 kb up and down stream of the gene)

In [ ]:
from pycisTopic.utils import region_names_to_coordinates, coord_to_region_names
from scenicplus.enhancer_to_gene import get_search_space
import pyranges as pr

max_dist = 150000
search_space = get_search_space(scenicpls_obj, pr_annot = pr.PyRanges(region_names_to_coordinates(scenicpls_obj.region_names.values)),
                                species = 'hsapiens',
                                assembly = 'hg38',
                                upstream = [1000, max_dist],
                                downstream = [1000, max_dist],
                                inplace=False,
                               )

2023-03-17 15:06:18,214 R2G          INFO     Downloading gene annotation from biomart dataset: hsapiens_gene_ensembl
2023-03-17 15:06:35,988 R2G          INFO     Downloading chromosome sizes from: http://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes
2023-03-17 15:06:41,401 R2G          INFO     Extending promoter annotation to 10 bp upstream and 10 downstream
2023-03-17 15:06:43,809 R2G          INFO     Extending search space to:
            						150000 bp downstream of the end of the gene.
            						150000 bp upstream of the start of the gene.
2023-03-17 15:06:54,975 R2G          INFO     Intersecting with regions.


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


2023-03-17 15:06:56,356 R2G          INFO     Calculating distances from region to gene
2023-03-17 15:09:39,076 R2G          INFO     Imploding multiple entries per region and gene


Building region to gene distance matrix

In [ ]:
search_space['Abs_distance'] = np.abs([x[0] for x in search_space['Distance'].to_list()]).astype(int)

In [ ]:
search_space_subset = search_space.loc[[gene in ad_rna.var_names for gene in search_space['Gene']]]
regions = search_space_subset.loc[:, 'Name'].unique().tolist()

In [ ]:
scenicpls_obj.subset(regions = regions)

In [ ]:
r2gdist = search_space_subset.pivot(index='Name', columns='Gene', values='Abs_distance')

In [ ]:
# Scale matrix between 0 and 1
r2gdist = r2gdist / max_dist
r2gdist = r2gdist.fillna(0)

In [ ]:
# Ensure same order of indices
regions = r2gdist.index.values.tolist()
genes = r2gdist.columns.values.tolist()

ad_atac = ad_atac[:, regions].copy()
ad_rna = ad_rna[:, genes].copy()

Make region to gene matrix sparse

In [ ]:
from scipy.sparse import coo_array, save_npz
r2gdist_coo = coo_array(r2gdist)

Import list of TFs

In [ ]:
TFs = pd.read_csv("/staging/leuven/stg_00002/lcb/gpartel/software/DeepSEM_multimodal/resources/allTFs_hg38.txt", header=None)[0].to_list()

In [ ]:
# Subset TFs present in RNA data
TFs = ad_rna.var_names[ad_rna.var_names.isin(TFs)]

SCENIC+ function to compute a TF to region scores used as prior

In [ ]:
from scenicplus.scenicplus_class import SCENICPLUS
from typing import List
import sys
sys.path.insert(0, "/staging/leuven/stg_00002/lcb/gpartel/software/DeepSEM_multimodal")
from src.object_converter import dict_motif_enrichment_results_to_mudata
from src.data_transformations import merge, get_max_rank_for_TF_to_region
import logging
import sys
import numba
import numpy as np

def calculate_TF_to_region_score(
    scplus_obj: SCENICPLUS,
    ctx_db_fname: str,
    annotations_to_use: List[str] = ['Orthology_annot', 'Direct_annot'],
    eRegulon_metadata_key: str = 'eRegulon_metadata',
    key_added: str = 'TF_to_region_max_rank') -> None:
    """
    Calculated TF-to-regions scores based on the maximum motif-rank-position for each TF and each region.
    Parameters
    ----------
    scplus_obj: SCENICPLUS
        A SCENIC+ object.
    ctx_db_fname: str
        Path to the cistarget ranking database used for motif enrichment analysis.
    annotations_to_use: List[str] = ['Orthology_annot', 'Direct_annot']
        List specifying which motif-to-TF annotations to use.
    eRegulon_metadata_key: str = 'eRegulon_metadata'
        Key in .uns under which to find the eRegulon metadata, scores will be stored in this dataframe
     key_added: str = 'TF_to_region_max_rank'
        Column name under which to store the TF-to-region scores.
    """
    level = logging.INFO
    format = '%(asctime)s %(name)-12s %(levelname)-8s %(message)s'
    handlers = [logging.StreamHandler(stream=sys.stdout)]
    logging.basicConfig(level=level, format=format, handlers=handlers)
    log = logging.getLogger('Triplet score')
    log.info("Converting motif enrichment dictionary (scplus_obj.menr) to AnnData ...")
    mdata_motifs = dict_motif_enrichment_results_to_mudata(scplus_obj.menr)
    var_names = mdata_motifs.var_names
    mdata_motifs.var_names_make_unique()
    adata_motifs = merge(mdata_motifs)
    adata_motifs.var_names = var_names
    log.info("Getting maximum motif-TF ranking for each TF and region.")
    adata_max_rank = get_max_rank_for_TF_to_region(
        adata_motifs, ctx_db_fname, annotations_to_use = annotations_to_use)
    df_max_rank = adata_max_rank.to_df()
    return df_max_rank


In [ ]:
TF_to_region_score = calculate_TF_to_region_score(scenicpls_obj, ctx_db_fname='/staging/leuven/stg_00002/lcb/sdewin/PhD/Multiomics_pipeline/analysis/pbmc_granulocyte_sorted_10k/consensus_database/v2_cluster_PBMC.regions_vs_motifs.rankings.feather')

In [ ]:
TF_to_region_score = TF_to_region_score.T
motif_df = pd.DataFrame(np.zeros((len(TFs), ad_atac.shape[1])), index= TFs, columns=ad_atac.var_names)
common_tfs = motif_df.index[motif_df.index.isin(TF_to_region_score.index)]
common_regs = motif_df.columns[motif_df.columns.isin(TF_to_region_score.columns)]
motif_df.loc[common_tfs, common_regs] = TF_to_region_score.loc[common_tfs, common_regs].values

Saving data

In [ ]:
ad_rna.write(DATA_PATH+'raw_exprMat.h5ad')
ad_atac.write(DATA_PATH+'fragment_matrix.h5ad')
save_npz(DATA_PATH+'r2gdist.npz', r2gdist_coo)
motif_df.to_pickle(DATA_PATH + '/TF2R_df.pkl')        
with open(DATA_PATH+'TFs.txt', 'w') as f:
    for item in TFs:
        f.write("%s\n" % item)